## Pillar 5 Recap
Simulate a "runaway background job" and practice the full diagnose → deprioritize → gracefully stop workflow, using tools that don't require sudo for the core loop.

**Steps**:

1. Launch two background "worker" processes with different simulated workloads:

In [ ]:
dennie@HP:~/linux-tut$ nice -n 15 sleep 400 &
[1] 43061
dennie@HP:~/linux-tut$ sleep 500 &
[2] 43088

2. Find both using `pgrep`, showing full command lines:

In [ ]:
dennie@HP:~/linux-tut$ pgrep -af sleep
43061 sleep 400
43088 sleep 500
43244 sleep 1

3. For the second one (the one not niced), check its current priority and demote it to the lowest priority tier without stopping it:

In [ ]:
dennie@HP:~/linux-tut$ ps -o pid,ni,cmd -p 43088
    PID  NI CMD
  43088   0 sleep 500

In [ ]:
dennie@HP:~/linux-tut$ renice -n 19 -p 43088
43088 (process ID) old priority 0, new priority 19

In [ ]:
dennie@HP:~/linux-tut$ ps -o pid,ni,cmd -p 43088
    PID  NI CMD
  43088  19 sleep 500

4. Confirm cron is active and check when it last ran (should be clean now after your cleanup, but this confirms cron didn't linger in a bad state from the practice):

In [ ]:
dennie@HP:~/linux-tut$ systemctl status cron
● cron.service - Regular background program processing daemon
     Loaded: loaded (/usr/lib/systemd/system/cron.service; enabled; preset: enabled)
     Active: active (running) since Sat 2026-09-12 07:35:04 UTC; 4h 19min ago
       Docs: man:cron(8)
   Main PID: 156 (cron)
      Tasks: 1 (limit: 9427)
     Memory: 448.0K (peak: 2.1M)
        CPU: 149ms
     CGroup: /system.slice/cron.service
             └─156 /usr/sbin/cron -f -P

Sep 12 08:17:04 HP CRON[9253]: pam_unix(cron:session): session closed for user root
Sep 12 09:17:04 HP CRON[18126]: pam_unix(cron:session): session opened for user root(uid=0) by root(uid=0)
Sep 12 09:17:04 HP CRON[18127]: (root) CMD (cd / && run-parts --report /etc/cron.hourly)
Sep 12 09:17:04 HP CRON[18126]: pam_unix(cron:session): session closed for user root
Sep 12 10:17:04 HP CRON[28964]: pam_unix(cron:session): session opened for user root(uid=0) by root(uid=0)
Sep 12 10:17:04 HP CRON[28965]: (root) CMD (cd / && run-parts --report /etc/cron.hourly)
Sep 12 10:17:04 HP CRON[28964]: pam_unix(cron:session): session closed for user root
Sep 12 11:17:04 HP CRON[38392]: pam_unix(cron:session): session opened for user root(uid=0) by root(uid=0)
Sep 12 11:17:04 HP CRON[38393]: (root) CMD (cd / && run-parts --report /etc/cron.hourly)
Sep 12 11:17:04 HP CRON[38392]: pam_unix(cron:session): session closed for user root

In [ ]:
dennie@HP:~/linux-tut$ journalctl -u cron -n 5 --no-pager
[2]+  Done                    sleep 500
Sep 12 13:17:04 HP CRON[59186]: (root) CMD(cd / && run-parts --report /etc/cron.hourly)
Sep 12 13:17:04 HP CRON[59185]: pam_unix(cron:session): session closed for user root
Sep 12 14:17:02 HP CRON[68995]: pam_unix(cron:session): session opened for user root(uid=0) by root(uid=0)
Sep 12 14:17:02 HP CRON[68996]: (root) CMD(cd / && run-parts --report /etc/cron.hourly)
Sep 12 14:17:02 HP CRON[68995]: pam_unix(cron:session): session closed for user root

5. Gracefully stop **both** sleep jobs using SIGTERM (not `-9`) via a single `pgrep`/`kill` combo:

In [ ]:
dennie@HP:~/linux-tut$ kill -15 $(pgrep sleep)

6. Verify everything is cleaned up:

In [ ]:
dennie@HP:~/linux-tut$ pgrep -a sleep
dennie@HP:~/linux-tut$ jobs